# Tools and Tool Descriptions

**What you will build:** an agent with several tools, and an understanding of the one thing that most controls whether it uses them well — the tool **description**. This maps to chapter 07 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/12_tools_and_descriptions.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Two ways to register a tool

- `@agent.tool_plain` — for a tool that needs nothing from the agent's context (most tools).
- `@agent.tool` — for a tool that needs the run context (`ctx`), e.g. to reach injected dependencies (that's notebook 1.3).

The function's **type hints** become the argument schema, and its **docstring** becomes the description the model reads. You write normal Python; PydanticAI turns it into a tool.

In [ ]:
agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@agent.tool_plain
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol, e.g. 'AAPL'."""
    fake = {"AAPL": 224.10, "GOOGL": 178.30, "TSLA": 251.44}
    return f"{ticker}: ${fake.get(ticker.upper(), 100.0)}"

@agent.tool_plain
def convert_currency(amount: float, to_currency: str) -> str:
    """Convert an amount in US dollars to another currency (EUR, GBP, JPY)."""
    rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 156.0}
    return f"{amount * rates.get(to_currency.upper(), 1.0):.2f} {to_currency.upper()}"

result = await agent.run("How much is 10 shares of AAPL worth in euros?")
print(result.output)

The agent chained two tools — get the price, then convert — with no routing code from you. It read the descriptions, decided the order, and combined the results.

## The description is the steering wheel

A tool's docstring is not documentation for humans — it's the instruction the model uses to decide **whether and when** to call the tool. Vague descriptions are the number-one cause of agents that ignore a tool or misuse it. Anthropic's research ([Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents)) puts tool descriptions at the top of what determines agent quality. Let's prove it.

In [ ]:
# A tool with a DELIBERATELY vague description:
vague = Agent(model, system_prompt="You are a helpful assistant.")

@vague.tool_plain
def lookup(x: str) -> str:
    """Does a thing with input."""          # <- useless description
    fake = {"AAPL": 224.10, "GOOGL": 178.30}
    return f"{x}: ${fake.get(x.upper(), 100.0)}"

r = await vague.run("What is the price of GOOGL?")
print(r.output)

# Which tools did the model actually call? Look for tool-call parts and print their names.
called = [p.tool_name for m in r.all_messages() for p in m.parts if getattr(p, 'part_kind', '') == 'tool-call']
print("\ntools the model actually called:", called or "(none — it skipped the tool)")

With a vague description the model often can't tell the tool is for stock prices, so it may skip it and guess (or refuse). Give the same function a clear docstring — `"Get the latest price for a stock ticker symbol."` — and it uses it reliably. You tune agent behavior far more through tool descriptions and the system prompt than through code.

## A real tool: live weather

Every tool so far returned fake data, on purpose — it keeps cells deterministic and free, a legitimate teaching choice. But real tools bring real failure modes: latency, timeouts, rate limits, malformed responses. Here's a genuinely live tool using [wttr.in](https://wttr.in), a free keyless weather API. Note the `try/except` — a real tool *will* fail sometimes, and the agent should get a useful error instead of crashing (the lesson from 0.3).

In [ ]:
import httpx   # already installed by pydantic-ai; no extra dependency

weather_agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@weather_agent.tool_plain
def live_weather(city: str) -> str:
    """Get the CURRENT real weather for a city, e.g. 'Bilbao'."""
    try:
        r = httpx.get(f"https://wttr.in/{city}?format=j1", timeout=10)
        cur = r.json()["current_condition"][0]
        return f"{cur['temp_C']}C, {cur['weatherDesc'][0]['value']}"
    except Exception as e:
        return f"Weather service unavailable: {e}"

result = await weather_agent.run("What's the weather like in Bilbao right now?")
print(result.output)

That was a real network round-trip — the model asked for `live_weather('Bilbao')`, your code called the API, and the result flowed back into the loop. The moment a tool touches the outside world, error handling stops being optional. Wrap it, set a timeout, and decide what the agent should see when it fails.

::::{dropdown} 🔍 What makes a good tool description
:color: info

```
Bad:   "Does a thing with input."
Good:  "Get the latest price for a stock ticker symbol, e.g. 'AAPL'. Use this whenever
        the user asks about a stock's current or recent price."
```

A good description says: **what** it does, **what the arguments mean** (with an example), and **when** to use it. Argument names and type hints matter too — `ticker: str` reads better to the model than `x: str`.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Fix the vague tool.** Rewrite `lookup`'s docstring to clearly say it returns stock prices, and rerun. Watch it become reliable.
2. **Add a units trap.** Add a `temperature(city: str)` tool that returns Celsius, and ask in Fahrenheit. See whether the model converts — then improve the description to tell it the unit.
3. **Name matters.** Rename `convert_currency`'s `to_currency` argument to `c` and see if reliability drops.
::::

::::{dropdown} 🔧 The "Think" tool
:color: info

A trick worth knowing: give the agent a tool that does **nothing** except let it write down its reasoning.

```python
@agent.tool_plain
def think(thought: str) -> str:
    """Use this to reason step by step before acting. It just records your thought."""
    return "noted"
```

On hard, multi-step tasks the model calls `think` to plan before calling real tools, which measurably improves reliability — many coding agents ship exactly this. It costs a turn, so use it only where planning actually helps. (The n8n course teaches the same tool in its tool-calling chapter.)
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **The model ignores a tool** | Description too vague, or unclear argument names | Rewrite the docstring to say what/when; use descriptive names + type hints |
| **Wrong tool chosen** | Two tools overlap in purpose | Make descriptions clearly distinct; say when NOT to use each |
| **Bad arguments passed** | Ambiguous argument meaning | Add an example to the docstring (e.g. "ticker like 'AAPL'") |
| **A real tool hangs the run** | No timeout on the network call | Always pass `timeout=` and wrap in `try/except` (see `live_weather`) |
::::

**What's next:** in **1.3** we add **typed output** and **dependency injection** — giving tools access to real data (a database, the current user) safely.